In [0]:
import os
import pyspark.sql.functions as F
from pyspark.sql.window import Window

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_DATABRICKS:
    catalog = "visagio_atividade_v_energy_360"
    silver_schema = f"{catalog}.silver"
    gold_schema = f"{catalog}.gold"
else:
    from local_runtime import setup_local_env
    spark, dbutils, display, _paths, _schemas = setup_local_env()
    catalog = _schemas["catalog"]
    silver_schema = _schemas["silver"]
    gold_schema = _schemas["gold"]

print(f"Iniciando a transformação para a camada Gold no schema: {gold_schema}")

df_uc_silver = spark.table(f"{silver_schema}.silver_unidades_consumidoras")
df_leituras_silver = spark.table(f"{silver_schema}.silver_leituras")
df_medidores_silver = spark.table(f"{silver_schema}.silver_medidores")
df_gd_silver = spark.table(f"{silver_schema}.silver_geracao_distribuida")

Iniciando a transformação para a camada Gold no schema: visagio_atividade_v_energy_360.gold


In [0]:
# Cell 2: Modelagem da dim_tempo
df_dim_tempo = df_leituras_silver.select("mes_referencia").distinct() \
    .filter(F.col("mes_referencia").isNotNull()) \
    .withColumn("competencia", F.date_format("mes_referencia", "yyyy-MM")) \
    .withColumn("ano", F.year("mes_referencia")) \
    .withColumn("mes", F.month("mes_referencia")) \
    .select("ano", "mes", "competencia")

In [0]:
# Cell 3: Modelagem da dim_unidade_consumidora

janela_medidor = Window.partitionBy("id_uc").orderBy(F.col("data_instalacao_medidor").desc_nulls_last())

df_medidor_ativo = df_medidores_silver \
    .filter(F.col("data_retirada").isNull() | (F.col("status") == "ATIVO")) \
    .withColumn("rn", F.row_number().over(janela_medidor)) \
    .filter("rn = 1") \
    .select("id_uc", F.col("id_medidor").alias("numero_medidor_ativo"))

df_dim_unidade_consumidora = df_uc_silver \
    .join(df_medidor_ativo, "id_uc", "left") \
    .join(df_gd_silver.select("id_uc", "data_conexao_gd"), "id_uc", "left") \
    .select(
        F.col("id_uc"),
        F.col("cpf_cnpj").alias("codigo_cliente"),
        F.col("nome_titular"),
        F.col("logradouro"),
        F.col("numero").cast("string"),
        F.col("bairro"),
        F.col("municipio"),
        F.col("uf"),
        F.col("cep"),
        F.col("classe").alias("classe_consumo"),
        F.col("subgrupo_tarifario"),
        F.col("alimentador"),
        F.lit(None).cast("string").alias("subestacao"),
        F.col("numero_medidor_ativo"),
        F.col("possui_geracao_solar").alias("possui_gd"),
        F.col("potencia_solar_kw").alias("potencia_gd_kw"),
        F.col("data_conexao_gd"),
        F.col("status")
    )

In [0]:
# Cell 4: Modelagem da fato_medicao
janela_leituras = Window.partitionBy("id_uc").orderBy("mes_referencia")

df_fato_medicao = df_leituras_silver \
    .withColumn("data_leitura_anterior", F.lag("data_leitura").over(janela_leituras)) \
    .filter(F.col("anomalia_leitura") == False) \
    .withColumn("dias_faturamento", F.datediff(F.col("data_leitura"), F.col("data_leitura_anterior"))) \
    .withColumn("dias_faturamento", 
                F.when((F.col("dias_faturamento") <= 0) | F.col("dias_faturamento").isNull(), F.lit(30))
                 .otherwise(F.col("dias_faturamento"))) \
    .withColumn("media_diaria_kwh", F.round(F.col("consumo_apurado_kwh") / F.col("dias_faturamento"), 4)) \
    .withColumn("competencia", F.date_format("mes_referencia", "yyyy-MM")) \
    .withColumn("id_medicao", F.concat_ws("_", F.col("id_uc"), F.col("competencia"))) \
    .withColumnRenamed("consumo_apurado_kwh", "consumo_kwh") \
    .select(
        "id_medicao",
        "id_uc",
        "id_medidor",
        "competencia",
        F.date_format("data_leitura", "yyyy-MM-dd").alias("data_leitura"),
        "leitura_anterior",
        "leitura_atual",
        "consumo_kwh",
        "dias_faturamento",
        "media_diaria_kwh",
        "tipo_leitura"
    )
df_fato_anomalia_leitura = df_leituras_silver \
    .filter(F.col("anomalia_leitura") == True) \
    .withColumn("competencia", F.date_format("mes_referencia", "yyyy-MM")) \
    .select(
        F.concat_ws("_", F.col("id_uc"), F.col("competencia")).alias("id_anomalia"),
        "id_uc", "id_medidor", "competencia",
        F.date_format("data_leitura", "yyyy-MM-dd").alias("data_leitura"),
        "leitura_anterior", "leitura_atual",
        (F.col("leitura_atual") - F.col("leitura_anterior")).alias("diferenca_leitura_kwh"),
        "tipo_leitura",
        F.lit("LEITURA_ATUAL_MENOR_SEM_ROLLOVER_OU_TROCA").alias("motivo_anomalia"))

In [0]:
# Cell 5: Gravação na Gold
print("Salvando as tabelas no Star Schema da Camada Gold...")

df_dim_tempo.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_tempo")

df_dim_unidade_consumidora.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_unidade_consumidora")

df_fato_medicao.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fato_medicao")

df_fato_anomalia_leitura.write.format("delta").mode("Overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fato_anomalia_leitura")

print("Fase 3: Modelagem Dimensional Gold concluída com sucesso.")

Salvando as tabelas no Star Schema da Camada Gold...
Fase 3: Modelagem Dimensional Gold concluída com sucesso.
